# Configurable Residual CBAM U-Net for Brain Tumor Segmentation



In [ ]:
# Cell 1 - Install requirements (Google Colab)

!pip install -q -U albumentations opencv-python-headless torchinfo scikit-image scipy

In [ ]:
# Cell 2 - Imports

import os
import gc
import json
import time
import random
import warnings
from copy import deepcopy
from pathlib import Path
from dataclasses import dataclass, asdict, replace
from typing import Dict, List, Optional, Tuple

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import albumentations as A
from albumentations.pytorch import ToTensorV2

warnings.filterwarnings("ignore")

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

In [ ]:
# Cell 3 - Reproducibility

def seed_everything(seed: int = 42) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Reproducible experiments. This may reduce speed slightly.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


SEED = 42
seed_everything(SEED)

print("Seed:", SEED)

In [ ]:
# Cell 3.5 - Mount Google Drive and define persistent paths

from google.colab import drive

drive.mount(
    "/content/drive"
)

DRIVE_PROJECT_ROOT = Path(
    "/content/drive/MyDrive/CBAM_UNet_Project"
)

DRIVE_DATA_ROOT = (
    DRIVE_PROJECT_ROOT
    / "data"
)

DRIVE_RESULTS_ROOT = (
    DRIVE_PROJECT_ROOT
    / "results"
)

DRIVE_DATA_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

DRIVE_RESULTS_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

print(
    "Project root:",
    DRIVE_PROJECT_ROOT,
)

print(
    "Data root:",
    DRIVE_DATA_ROOT,
)

print(
    "Results root:",
    DRIVE_RESULTS_ROOT,
)

In [ ]:
# Cell 4 - Download dataset from Kaggle

# Before running this cell in Colab, place kaggle.json in:
# /content/drive/MyDrive/kaggle/kaggle.json
#
# Then mount Google Drive if needed:
# from google.colab import drive
# drive.mount("/content/drive")

DATA_ROOT = Path("/content/lgg-mri-segmentation/kaggle_3m")

if not DATA_ROOT.exists():
    os.environ["KAGGLE_CONFIG_DIR"] = "/content/drive/MyDrive/kaggle"

    !kaggle datasets download -d mateuszbuda/lgg-mri-segmentation
    !unzip -q -o lgg-mri-segmentation.zip -d /content

assert DATA_ROOT.exists(), f"Dataset directory not found: {DATA_ROOT}"

print("Dataset root:", DATA_ROOT)

In [ ]:
# Cell 5 - Global and experiment configurations

@dataclass
class GlobalConfig:
    data_root: str = str(DATA_ROOT)
    image_size: int = 256
    in_channels: int = 3
    num_classes: int = 1

    # Set 32 for a lighter model or 64 for the original larger model.
    base_channels: int = 32

    batch_size: int = 8
    epochs: int = 50
    learning_rate: float = 1e-4
    weight_decay: float = 1e-5

    num_workers: int = 2
    cache_images: bool = True
    use_weighted_sampler: bool = True

    use_amp: bool = True
    gradient_clip: float = 1.0

    early_stopping_patience: int = 10
    scheduler_patience: int = 3
    scheduler_factor: float = 0.5

    threshold: float = 0.5
    seed: int = 42

    results_dir: str = str(
    DRIVE_RESULTS_ROOT
    )
    device: str = (
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )


@dataclass
class ExperimentConfig:
    name: str

    use_residual: bool = True
    use_cbam: bool = True
    use_augmentation: bool = True

    use_aspp: bool = True
    use_deep_supervision: bool = True

    # Supported values:
    # "dice_focal"
    # "dice_focal_tversky"
    loss_name: str = "dice_focal"


cfg = GlobalConfig()

Path(cfg.results_dir).mkdir(
    parents=True,
    exist_ok=True,
)

print(asdict(cfg))

In [ ]:
# Cell 5.5 - Verify persistent storage

results_directory = Path(
    cfg.results_dir
)

if not results_directory.exists():
    raise FileNotFoundError(
        f"Results directory does not exist: "
        f"{results_directory}"
    )

test_file = (
    results_directory
    / "_drive_write_test.txt"
)

test_file.write_text(
    "Google Drive persistence test passed.",
    encoding="utf-8",
)

if not test_file.exists():
    raise RuntimeError(
        "Unable to write to Google Drive."
    )

print(
    "Persistent results directory:",
    results_directory,
)

print(
    "Google Drive write test passed."
)

In [ ]:
# Cell 6 - Save environment information

environment = {
    "torch": torch.__version__,
    "cuda_version": torch.version.cuda,
    "cudnn_version": torch.backends.cudnn.version(),
    "device": cfg.device,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}

with open(Path(cfg.results_dir) / "environment.json", "w") as file:
    json.dump(environment, file, indent=4)

environment

In [ ]:
# Cell 7 - Scan image and mask paths

def build_dataframe(root: str) -> pd.DataFrame:
    root = Path(root)
    records = []

    patient_dirs = sorted([path for path in root.iterdir() if path.is_dir()])

    for patient_dir in tqdm(patient_dirs, desc="Scanning patients"):
        for image_path in sorted(patient_dir.glob("*.tif")):
            if image_path.stem.endswith("_mask"):
                continue

            mask_path = image_path.with_name(f"{image_path.stem}_mask.tif")

            if mask_path.exists():
                records.append({
                    "patient": patient_dir.name,
                    "image": str(image_path),
                    "mask": str(mask_path),
                })

    dataframe = pd.DataFrame(records)

    if dataframe.empty:
        raise RuntimeError(f"No valid image-mask pairs found under {root}")

    return dataframe


df = build_dataframe(cfg.data_root)

print("Images:", len(df))
print("Patients:", df["patient"].nunique())

df.head()

In [ ]:
# Cell 8 - Read mask statistics safely

def read_mask_information(mask_path: str) -> Dict:
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

    if mask is None:
        raise FileNotFoundError(f"Could not read mask: {mask_path}")

    height, width = mask.shape
    area = int((mask > 0).sum())
    ratio = float(area / (height * width))

    return {
        "mask_area": area,
        "mask_ratio": ratio,
        "height": height,
        "width": width,
        "has_tumor": int(area > 0),
    }


mask_information = [
    read_mask_information(path)
    for path in tqdm(df["mask"], desc="Reading masks")
]

mask_information = pd.DataFrame(mask_information)
df = pd.concat([df.reset_index(drop=True), mask_information], axis=1)

df.head()

In [ ]:
# Cell 9 - Define tumor-size groups using relative mask area

def tumor_size_from_ratio(mask_ratio: float) -> str:
    if mask_ratio == 0:
        return "none"
    if mask_ratio < 0.005:
        return "small"
    if mask_ratio < 0.03:
        return "medium"
    return "large"


df["tumor_size"] = df["mask_ratio"].apply(tumor_size_from_ratio)

display(
    df["tumor_size"]
    .value_counts()
    .rename_axis("tumor_size")
    .reset_index(name="count")
)

In [ ]:
# Cell 10 - Patient-level stratification information

patient_summary = (
    df.groupby("patient")
      .agg(
          image_count=("image", "size"),
          tumor_image_count=("has_tumor", "sum"),
          tumor_ratio=("has_tumor", "mean"),
          total_tumor_area=("mask_area", "sum"),
      )
      .reset_index()
)

# Quantile-based burden groups. duplicates='drop' handles repeated values.
try:
    patient_summary["burden_group"] = pd.qcut(
        patient_summary["tumor_ratio"],
        q=4,
        labels=False,
        duplicates="drop",
    ).astype(str)
except ValueError:
    patient_summary["burden_group"] = (
        patient_summary["tumor_ratio"] > 0
    ).astype(int).astype(str)

patient_summary.head()

In [ ]:
# Cell 11 - Patient-level train/validation/test split

def patient_level_split(
    dataframe: pd.DataFrame,
    patient_table: pd.DataFrame,
    seed: int = 42,
    train_fraction: float = 0.80,
    validation_fraction: float = 0.10,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:

    patients = patient_table["patient"].values
    strata = patient_table["burden_group"].values

    try:
        train_patients, temporary_patients = train_test_split(
            patients,
            train_size=train_fraction,
            random_state=seed,
            stratify=strata,
        )
    except ValueError:
        train_patients, temporary_patients = train_test_split(
            patients,
            train_size=train_fraction,
            random_state=seed,
            shuffle=True,
        )

    temporary_table = patient_table[
        patient_table["patient"].isin(temporary_patients)
    ].reset_index(drop=True)

    relative_validation_fraction = validation_fraction / (1 - train_fraction)

    try:
        validation_patients, test_patients = train_test_split(
            temporary_table["patient"].values,
            train_size=relative_validation_fraction,
            random_state=seed,
            stratify=temporary_table["burden_group"].values,
        )
    except ValueError:
        validation_patients, test_patients = train_test_split(
            temporary_table["patient"].values,
            train_size=relative_validation_fraction,
            random_state=seed,
            shuffle=True,
        )

    train_df = dataframe[
        dataframe["patient"].isin(train_patients)
    ].reset_index(drop=True)

    validation_df = dataframe[
        dataframe["patient"].isin(validation_patients)
    ].reset_index(drop=True)

    test_df = dataframe[
        dataframe["patient"].isin(test_patients)
    ].reset_index(drop=True)

    return train_df, validation_df, test_df


train_df, validation_df, test_df = patient_level_split(
    dataframe=df,
    patient_table=patient_summary,
    seed=cfg.seed,
)

print("Split completed.")

In [ ]:
# Cell 12 - Leakage checks and split report

assert set(train_df["patient"]).isdisjoint(validation_df["patient"])
assert set(train_df["patient"]).isdisjoint(test_df["patient"])
assert set(validation_df["patient"]).isdisjoint(test_df["patient"])

def split_report(name: str, dataframe: pd.DataFrame) -> Dict:
    return {
        "split": name,
        "patients": dataframe["patient"].nunique(),
        "images": len(dataframe),
        "tumor_images": int(dataframe["has_tumor"].sum()),
        "tumor_image_percent": 100 * dataframe["has_tumor"].mean(),
        "mean_mask_ratio": dataframe["mask_ratio"].mean(),
    }


split_table = pd.DataFrame([
    split_report("train", train_df),
    split_report("validation", validation_df),
    split_report("test", test_df),
])

display(split_table.round(4))
print("No patient leakage.")

In [ ]:
# Cell 13 - Save fixed splits only when they do not already exist

split_directory = (
    Path(cfg.results_dir)
    / "splits"
)

split_directory.mkdir(
    parents=True,
    exist_ok=True,
)

train_split_path = (
    split_directory
    / "train.csv"
)

validation_split_path = (
    split_directory
    / "validation.csv"
)

test_split_path = (
    split_directory
    / "test.csv"
)

if not (
    train_split_path.exists()
    and validation_split_path.exists()
    and test_split_path.exists()
):
    train_df.to_csv(
        train_split_path,
        index=False,
    )

    validation_df.to_csv(
        validation_split_path,
        index=False,
    )

    test_df.to_csv(
        test_split_path,
        index=False,
    )

    print(
        "Fixed splits were saved to:",
        split_directory,
    )

else:
    print(
        "Existing fixed splits were preserved:",
        split_directory,
    )

In [ ]:
# Cell 13.5 - Load persistent fixed splits when available

split_directory = (
    Path(cfg.results_dir)
    / "splits"
)

train_split_path = (
    split_directory
    / "train.csv"
)

validation_split_path = (
    split_directory
    / "validation.csv"
)

test_split_path = (
    split_directory
    / "test.csv"
)


if (
    train_split_path.exists()
    and validation_split_path.exists()
    and test_split_path.exists()
):

    train_df = pd.read_csv(
        train_split_path
    )

    validation_df = pd.read_csv(
        validation_split_path
    )

    test_df = pd.read_csv(
        test_split_path
    )

    print(
        "Fixed dataset splits were loaded "
        "from Google Drive."
    )

    print(
        "Train samples:",
        len(train_df),
    )

    print(
        "Validation samples:",
        len(validation_df),
    )

    print(
        "Test samples:",
        len(test_df),
    )

else:

    raise FileNotFoundError(
        "One or more fixed split files "
        "were not found."
    )

In [ ]:
# Cell 14 - Transform factory

def get_transforms(
    image_size: int,
    use_augmentation: bool,
) -> Tuple[A.Compose, A.Compose]:

    evaluation_transform = A.Compose([
        A.Resize(image_size, image_size),
        A.Normalize(),
        ToTensorV2(),
    ])

    if not use_augmentation:
        return evaluation_transform, evaluation_transform

    training_transform = A.Compose([
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.1),

        A.OneOf([
            A.RandomRotate90(p=1.0),
            A.Affine(
                scale=(0.95, 1.05),
                translate_percent=(-0.03, 0.03),
                rotate=(-10, 10),
                shear=(-3, 3),
                p=1.0,
            ),
        ], p=0.5),

        A.ElasticTransform(
            alpha=10,
            sigma=5,
            p=0.1,
        ),

        A.OneOf([
            A.GaussNoise(std_range=(0.01, 0.05), p=1.0),
            A.GaussianBlur(blur_limit=(3, 5), p=1.0),
            A.MotionBlur(blur_limit=(3, 5), p=1.0),
        ], p=0.2),

        A.OneOf([
            A.RandomBrightnessContrast(
                brightness_limit=0.1,
                contrast_limit=0.1,
                p=1.0,
            ),
            A.RandomGamma(gamma_limit=(90, 110), p=1.0),
            A.CLAHE(clip_limit=2.0, p=1.0),
        ], p=0.25),

        A.Resize(image_size, image_size),
        A.Normalize(),
        ToTensorV2(),
    ])

    return training_transform, evaluation_transform

In [ ]:
# Cell 15 - Dataset

class BrainTumorDataset(Dataset):
    def __init__(
        self,
        dataframe: pd.DataFrame,
        transform: Optional[A.Compose] = None,
        cache_images: bool = False,
    ):
        self.dataframe = dataframe.reset_index(drop=True).copy()
        self.transform = transform
        self.cache_images = cache_images
        self.cache: Dict[int, Tuple[np.ndarray, np.ndarray]] = {}

        if self.cache_images:
            self._build_cache()

    def _read_pair(self, index: int) -> Tuple[np.ndarray, np.ndarray]:
        row = self.dataframe.iloc[index]

        image = cv2.imread(row["image"], cv2.IMREAD_COLOR)
        mask = cv2.imread(row["mask"], cv2.IMREAD_GRAYSCALE)

        if image is None:
            raise FileNotFoundError(f"Could not read image: {row['image']}")

        if mask is None:
            raise FileNotFoundError(f"Could not read mask: {row['mask']}")

        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mask = (mask > 0).astype(np.uint8)

        return image, mask

    def _build_cache(self) -> None:
        for index in tqdm(
            range(len(self.dataframe)),
            desc="Caching images",
            leave=False,
        ):
            self.cache[index] = self._read_pair(index)

    def __len__(self) -> int:
        return len(self.dataframe)

    def __getitem__(self, index: int) -> Dict:
        if self.cache_images:
            image, mask = self.cache[index]

            # Albumentations may modify arrays in place.
            image = image.copy()
            mask = mask.copy()
        else:
            image, mask = self._read_pair(index)

        if self.transform is not None:
            transformed = self.transform(image=image, mask=mask)
            image = transformed["image"]
            mask = transformed["mask"]

        mask = mask.float().unsqueeze(0)

        row = self.dataframe.iloc[index]

        return {
            "image": image.float(),
            "mask": mask,
            "has_tumor": torch.tensor(row["has_tumor"], dtype=torch.bool),
            "mask_ratio": torch.tensor(row["mask_ratio"], dtype=torch.float32),
            "patient": row["patient"],
            "image_path": row["image"],
        }

In [ ]:
# Cell 16 - Weighted sampler and DataLoader factory

SAMPLER_WEIGHTS = {
    "none": 1.0,
    "small": 3.0,
    "medium": 2.0,
    "large": 1.5,
}


def create_weighted_sampler(dataframe: pd.DataFrame) -> WeightedRandomSampler:
    sample_weights = (
        dataframe["tumor_size"]
        .map(SAMPLER_WEIGHTS)
        .astype(np.float64)
        .values
    )

    return WeightedRandomSampler(
        weights=torch.as_tensor(sample_weights, dtype=torch.double),
        num_samples=len(sample_weights),
        replacement=True,
    )


def build_dataloaders(
    experiment: ExperimentConfig,
) -> Tuple[DataLoader, DataLoader, DataLoader]:

    train_transform, evaluation_transform = get_transforms(
        image_size=cfg.image_size,
        use_augmentation=experiment.use_augmentation,
    )

    train_dataset = BrainTumorDataset(
        dataframe=train_df,
        transform=train_transform,
        cache_images=cfg.cache_images,
    )

    validation_dataset = BrainTumorDataset(
        dataframe=validation_df,
        transform=evaluation_transform,
        cache_images=cfg.cache_images,
    )

    test_dataset = BrainTumorDataset(
        dataframe=test_df,
        transform=evaluation_transform,
        cache_images=cfg.cache_images,
    )

    sampler = (
        create_weighted_sampler(train_df)
        if cfg.use_weighted_sampler
        else None
    )

    loader_arguments = {
        "batch_size": cfg.batch_size,
        "num_workers": cfg.num_workers,
        "pin_memory": torch.cuda.is_available(),
        "persistent_workers": cfg.num_workers > 0,
    }

    train_loader = DataLoader(
        train_dataset,
        sampler=sampler,
        shuffle=sampler is None,
        drop_last=False,
        **loader_arguments,
    )

    validation_loader = DataLoader(
        validation_dataset,
        shuffle=False,
        drop_last=False,
        **loader_arguments,
    )

    test_loader = DataLoader(
        test_dataset,
        shuffle=False,
        drop_last=False,
        **loader_arguments,
    )

    return train_loader, validation_loader, test_loader

In [ ]:
# Cell 17 - Visual sanity check

preview_experiment = ExperimentConfig(
    name="preview",
    use_residual=True,
    use_cbam=True,
    use_augmentation=True,
    use_aspp=True,
    use_deep_supervision=True,
)

preview_train_loader, _, _ = build_dataloaders(preview_experiment)
preview_batch = next(iter(preview_train_loader))

images = preview_batch["image"][:4]
masks = preview_batch["mask"][:4]

figure, axes = plt.subplots(2, 4, figsize=(14, 7))

for index in range(4):
    image = images[index].permute(1, 2, 0).numpy()
    image = (image - image.min()) / (image.max() - image.min() + 1e-8)

    axes[0, index].imshow(image)
    axes[0, index].axis("off")
    axes[0, index].set_title("MRI")

    axes[1, index].imshow(masks[index, 0], cmap="gray")
    axes[1, index].axis("off")
    axes[1, index].set_title("Mask")

plt.tight_layout()
plt.show()

del preview_train_loader, preview_batch, images, masks
gc.collect()

## Model components

در حالت خاموش بودن Residual، بلاک استاندارد double-convolution استفاده می‌شود.  
در حالت خاموش بودن CBAM یا ASPP، از `nn.Identity` یا bottleneck ساده استفاده می‌شود؛ بنابراین مسیر محاسباتی معتبر باقی می‌ماند.

In [ ]:
# Cell 18 - Normalization and convolution blocks

def normalization_layer(channels: int) -> nn.Module:
    # GroupNorm is stable for small segmentation batch sizes.
    groups = min(8, channels)

    while channels % groups != 0:
        groups -= 1

    return nn.GroupNorm(groups, channels)


class ConvBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False,
            ),
            normalization_layer(out_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(
                out_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False,
            ),
            normalization_layer(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class ResidualBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()

        self.main = nn.Sequential(
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False,
            ),
            normalization_layer(out_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(
                out_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False,
            ),
            normalization_layer(out_channels),
        )

        self.shortcut = (
            nn.Identity()
            if in_channels == out_channels
            else nn.Sequential(
                nn.Conv2d(
                    in_channels,
                    out_channels,
                    kernel_size=1,
                    bias=False,
                ),
                normalization_layer(out_channels),
            )
        )

        self.activation = nn.ReLU(inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.activation(self.main(x) + self.shortcut(x))


def build_feature_block(
    in_channels: int,
    out_channels: int,
    use_residual: bool,
) -> nn.Module:
    block_class = ResidualBlock if use_residual else ConvBlock
    return block_class(in_channels, out_channels)

In [ ]:
# Cell 19 - CBAM

class ChannelAttention(nn.Module):
    def __init__(self, channels: int, reduction: int = 16):
        super().__init__()

        hidden_channels = max(channels // reduction, 1)

        self.shared_mlp = nn.Sequential(
            nn.Conv2d(channels, hidden_channels, kernel_size=1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden_channels, channels, kernel_size=1, bias=False),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        average_attention = self.shared_mlp(
            F.adaptive_avg_pool2d(x, output_size=1)
        )

        maximum_attention = self.shared_mlp(
            F.adaptive_max_pool2d(x, output_size=1)
        )

        attention = torch.sigmoid(
            average_attention + maximum_attention
        )

        return x * attention


class SpatialAttention(nn.Module):
    def __init__(self, kernel_size: int = 7):
        super().__init__()

        padding = kernel_size // 2

        self.convolution = nn.Conv2d(
            2,
            1,
            kernel_size=kernel_size,
            padding=padding,
            bias=False,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        average_map = torch.mean(x, dim=1, keepdim=True)
        maximum_map = torch.amax(x, dim=1, keepdim=True)

        attention = torch.sigmoid(
            self.convolution(
                torch.cat([average_map, maximum_map], dim=1)
            )
        )

        return x * attention


class CBAM(nn.Module):
    def __init__(
        self,
        channels: int,
        reduction: int = 16,
        spatial_kernel_size: int = 7,
    ):
        super().__init__()

        self.channel_attention = ChannelAttention(
            channels=channels,
            reduction=reduction,
        )

        self.spatial_attention = SpatialAttention(
            kernel_size=spatial_kernel_size,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.channel_attention(x)
        x = self.spatial_attention(x)
        return x


def build_attention(
    channels: int,
    use_cbam: bool,
) -> nn.Module:
    return CBAM(channels) if use_cbam else nn.Identity()

In [ ]:
# Cell 20 - ASPP

class ASPPConv(nn.Sequential):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        dilation: int,
    ):
        super().__init__(
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                padding=dilation,
                dilation=dilation,
                bias=False,
            ),
            normalization_layer(out_channels),
            nn.ReLU(inplace=True),
        )


class ASPP(nn.Module):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        dilation_rates: Tuple[int, ...] = (1, 2, 4, 6),
    ):
        super().__init__()

        branch_channels = max(out_channels // len(dilation_rates), 1)

        self.branches = nn.ModuleList([
            ASPPConv(
                in_channels=in_channels,
                out_channels=branch_channels,
                dilation=rate,
            )
            for rate in dilation_rates
        ])

        merged_channels = branch_channels * len(dilation_rates)

        self.project = nn.Sequential(
            nn.Conv2d(
                merged_channels,
                out_channels,
                kernel_size=1,
                bias=False,
            ),
            normalization_layer(out_channels),
            nn.ReLU(inplace=True),
            nn.Dropout2d(p=0.1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        features = [branch(x) for branch in self.branches]
        return self.project(torch.cat(features, dim=1))

In [ ]:
# Cell 21 - Encoder and decoder blocks

class EncoderBlock(nn.Module):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        use_residual: bool,
        use_cbam: bool,
    ):
        super().__init__()

        self.features = build_feature_block(
            in_channels,
            out_channels,
            use_residual,
        )

        self.attention = build_attention(
            out_channels,
            use_cbam,
        )

        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

    def forward(
        self,
        x: torch.Tensor,
    ) -> Tuple[torch.Tensor, torch.Tensor]:

        features = self.attention(self.features(x))
        pooled = self.pool(features)

        return features, pooled


class DecoderBlock(nn.Module):
    def __init__(
        self,
        in_channels: int,
        skip_channels: int,
        out_channels: int,
        use_residual: bool,
        use_cbam: bool,
    ):
        super().__init__()

        self.upsample = nn.ConvTranspose2d(
            in_channels,
            out_channels,
            kernel_size=2,
            stride=2,
        )

        self.features = build_feature_block(
            out_channels + skip_channels,
            out_channels,
            use_residual,
        )

        self.attention = build_attention(
            out_channels,
            use_cbam,
        )

    def forward(
        self,
        x: torch.Tensor,
        skip: torch.Tensor,
    ) -> torch.Tensor:

        x = self.upsample(x)

        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(
                x,
                size=skip.shape[-2:],
                mode="bilinear",
                align_corners=False,
            )

        x = torch.cat([skip, x], dim=1)
        x = self.features(x)
        x = self.attention(x)

        return x

In [ ]:
# Cell 22 - Configurable Residual CBAM U-Net

class ConfigurableCBAMUNet(nn.Module):
    def __init__(
        self,
        in_channels: int = 3,
        num_classes: int = 1,
        base_channels: int = 32,
        use_residual: bool = True,
        use_cbam: bool = True,
        use_aspp: bool = True,
        use_deep_supervision: bool = True,
    ):
        super().__init__()

        self.use_deep_supervision = use_deep_supervision

        c1 = base_channels
        c2 = base_channels * 2
        c3 = base_channels * 4
        c4 = base_channels * 8
        c5 = base_channels * 16

        self.encoder1 = EncoderBlock(
            in_channels, c1, use_residual, use_cbam
        )
        self.encoder2 = EncoderBlock(
            c1, c2, use_residual, use_cbam
        )
        self.encoder3 = EncoderBlock(
            c2, c3, use_residual, use_cbam
        )
        self.encoder4 = EncoderBlock(
            c3, c4, use_residual, use_cbam
        )

        if use_aspp:
            self.bottleneck = ASPP(
                in_channels=c4,
                out_channels=c5,
            )
        else:
            self.bottleneck = build_feature_block(
                c4,
                c5,
                use_residual,
            )

        self.bottleneck_attention = build_attention(
            c5,
            use_cbam,
        )

        self.decoder4 = DecoderBlock(
            c5, c4, c4, use_residual, use_cbam
        )
        self.decoder3 = DecoderBlock(
            c4, c3, c3, use_residual, use_cbam
        )
        self.decoder2 = DecoderBlock(
            c3, c2, c2, use_residual, use_cbam
        )
        self.decoder1 = DecoderBlock(
            c2, c1, c1, use_residual, use_cbam
        )

        self.output_head = nn.Conv2d(
            c1,
            num_classes,
            kernel_size=1,
        )

        if self.use_deep_supervision:
            self.deep_head_3 = nn.Conv2d(
                c3,
                num_classes,
                kernel_size=1,
            )
            self.deep_head_2 = nn.Conv2d(
                c2,
                num_classes,
                kernel_size=1,
            )

    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        input_size = x.shape[-2:]

        skip1, x = self.encoder1(x)
        skip2, x = self.encoder2(x)
        skip3, x = self.encoder3(x)
        skip4, x = self.encoder4(x)

        x = self.bottleneck(x)
        x = self.bottleneck_attention(x)

        x = self.decoder4(x, skip4)
        x = self.decoder3(x, skip3)
        deep_feature_3 = x

        x = self.decoder2(x, skip2)
        deep_feature_2 = x

        x = self.decoder1(x, skip1)
        main_output = self.output_head(x)

        outputs = {"main": main_output}

        if self.use_deep_supervision:
            outputs["deep_2"] = F.interpolate(
                self.deep_head_2(deep_feature_2),
                size=input_size,
                mode="bilinear",
                align_corners=False,
            )

            outputs["deep_3"] = F.interpolate(
                self.deep_head_3(deep_feature_3),
                size=input_size,
                mode="bilinear",
                align_corners=False,
            )

        return outputs

In [ ]:
# Cell 23 - Model factory and forward test

def build_model(experiment: ExperimentConfig) -> nn.Module:
    return ConfigurableCBAMUNet(
        in_channels=cfg.in_channels,
        num_classes=cfg.num_classes,
        base_channels=cfg.base_channels,
        use_residual=experiment.use_residual,
        use_cbam=experiment.use_cbam,
        use_aspp=experiment.use_aspp,
        use_deep_supervision=experiment.use_deep_supervision,
    )


test_experiment = ExperimentConfig(
    name="forward_test",
    use_residual=True,
    use_cbam=True,
    use_augmentation=True,
    use_aspp=True,
    use_deep_supervision=True,
)

test_model = build_model(test_experiment).to(cfg.device)

test_input = torch.randn(
    2,
    cfg.in_channels,
    cfg.image_size,
    cfg.image_size,
    device=cfg.device,
)

with torch.no_grad():
    test_outputs = test_model(test_input)

print({
    key: tuple(value.shape)
    for key, value in test_outputs.items()
})

print(
    "Parameters:",
    f"{sum(parameter.numel() for parameter in test_model.parameters()):,}",
)

del test_model, test_input, test_outputs
torch.cuda.empty_cache()

## Loss

ترکیب نهایی از Dice و Focal استفاده می‌کند.  
Tversky به‌صورت اختیاری در کلاس موجود است، ولی برای جلوگیری از هم‌پوشانی بیش از حد lossها، در تنظیم پیش‌فرض وارد total loss نشده است.

In [ ]:
# Cell 24 - Segmentation losses

class DiceLoss(nn.Module):
    def __init__(self, smooth: float = 1.0):
        super().__init__()
        self.smooth = smooth

    def forward(
        self,
        logits: torch.Tensor,
        targets: torch.Tensor,
    ) -> torch.Tensor:

        probabilities = torch.sigmoid(logits)

        probabilities = probabilities.flatten(start_dim=1)
        targets = targets.flatten(start_dim=1)

        intersection = (probabilities * targets).sum(dim=1)
        denominator = probabilities.sum(dim=1) + targets.sum(dim=1)

        dice = (
            2 * intersection + self.smooth
        ) / (
            denominator + self.smooth
        )

        return 1 - dice.mean()


class BinaryFocalLoss(nn.Module):
    def __init__(
        self,
        alpha: float = 0.8,
        gamma: float = 2.0,
    ):
        super().__init__()

        self.alpha = alpha
        self.gamma = gamma

    def forward(
        self,
        logits: torch.Tensor,
        targets: torch.Tensor,
    ) -> torch.Tensor:

        binary_cross_entropy = F.binary_cross_entropy_with_logits(
            logits,
            targets,
            reduction="none",
        )

        probability_correct = torch.exp(-binary_cross_entropy)

        focal_loss = (
            self.alpha
            * (1 - probability_correct).pow(self.gamma)
            * binary_cross_entropy
        )

        return focal_loss.mean()


class TverskyLoss(nn.Module):
    def __init__(
        self,
        alpha: float = 0.3,
        beta: float = 0.7,
        smooth: float = 1.0,
    ):
        super().__init__()

        self.alpha = alpha
        self.beta = beta
        self.smooth = smooth

    def forward(
        self,
        logits: torch.Tensor,
        targets: torch.Tensor,
    ) -> torch.Tensor:

        probabilities = torch.sigmoid(logits)

        probabilities = probabilities.flatten(start_dim=1)
        targets = targets.flatten(start_dim=1)

        true_positive = (probabilities * targets).sum(dim=1)
        false_positive = (
            probabilities * (1 - targets)
        ).sum(dim=1)
        false_negative = (
            (1 - probabilities) * targets
        ).sum(dim=1)

        score = (
            true_positive + self.smooth
        ) / (
            true_positive
            + self.alpha * false_positive
            + self.beta * false_negative
            + self.smooth
        )

        return 1 - score.mean()

In [ ]:
# Cell 25 - Configurable segmentation loss
# Baseline: Dice + Focal
# Recall-oriented: Dice + Focal + Tversky


class DiceFocalSegmentationLoss(nn.Module):
    def __init__(
        self,
        dice_weight: float = 0.50,
        focal_weight: float = 0.50,
    ):
        super().__init__()

        self.dice_weight = dice_weight
        self.focal_weight = focal_weight

        total_weight = (
            dice_weight
            + focal_weight
        )

        if not np.isclose(
            total_weight,
            1.0,
        ):
            raise ValueError(
                "Dice and Focal weights must sum "
                f"to 1.0, but received {total_weight:.4f}."
            )

        self.dice_loss = DiceLoss(
            smooth=1.0,
        )

        self.focal_loss = BinaryFocalLoss(
            alpha=0.8,
            gamma=2.0,
        )

    def single_output_loss(
        self,
        logits: torch.Tensor,
        targets: torch.Tensor,
    ) -> torch.Tensor:

        dice_value = self.dice_loss(
            logits,
            targets,
        )

        focal_value = self.focal_loss(
            logits,
            targets,
        )

        return (
            self.dice_weight * dice_value
            + self.focal_weight * focal_value
        )

    def forward(
        self,
        outputs: Dict[str, torch.Tensor],
        targets: torch.Tensor,
    ) -> torch.Tensor:

        main_loss = self.single_output_loss(
            outputs["main"],
            targets,
        )

        if (
            "deep_2" in outputs
            and "deep_3" in outputs
        ):
            deep_2_loss = self.single_output_loss(
                outputs["deep_2"],
                targets,
            )

            deep_3_loss = self.single_output_loss(
                outputs["deep_3"],
                targets,
            )

            return (
                0.70 * main_loss
                + 0.20 * deep_2_loss
                + 0.10 * deep_3_loss
            )

        if "deep_2" in outputs:
            deep_2_loss = self.single_output_loss(
                outputs["deep_2"],
                targets,
            )

            return (
                0.80 * main_loss
                + 0.20 * deep_2_loss
            )

        if "deep_3" in outputs:
            deep_3_loss = self.single_output_loss(
                outputs["deep_3"],
                targets,
            )

            return (
                0.90 * main_loss
                + 0.10 * deep_3_loss
            )

        return main_loss


class DiceFocalTverskySegmentationLoss(nn.Module):
    def __init__(
        self,
        dice_weight: float = 0.25,
        focal_weight: float = 0.15,
        tversky_weight: float = 0.60,
        tversky_alpha: float = 0.30,
        tversky_beta: float = 0.70,
    ):
        super().__init__()

        self.dice_weight = dice_weight
        self.focal_weight = focal_weight
        self.tversky_weight = tversky_weight

        total_weight = (
            dice_weight
            + focal_weight
            + tversky_weight
        )

        if not np.isclose(
            total_weight,
            1.0,
        ):
            raise ValueError(
                "Dice, Focal, and Tversky weights "
                f"must sum to 1.0, but received {total_weight:.4f}."
            )

        self.dice_loss = DiceLoss(
            smooth=1.0,
        )

        self.focal_loss = BinaryFocalLoss(
            alpha=0.8,
            gamma=2.0,
        )

        # beta > alpha assigns a larger penalty
        # to false-negative tumor pixels.
        self.tversky_loss = TverskyLoss(
            alpha=tversky_alpha,
            beta=tversky_beta,
            smooth=1.0,
        )

    def single_output_loss(
        self,
        logits: torch.Tensor,
        targets: torch.Tensor,
    ) -> torch.Tensor:

        dice_value = self.dice_loss(
            logits,
            targets,
        )

        focal_value = self.focal_loss(
            logits,
            targets,
        )

        tversky_value = self.tversky_loss(
            logits,
            targets,
        )

        return (
            self.dice_weight * dice_value
            + self.focal_weight * focal_value
            + self.tversky_weight * tversky_value
        )

    def forward(
        self,
        outputs: Dict[str, torch.Tensor],
        targets: torch.Tensor,
    ) -> torch.Tensor:

        main_loss = self.single_output_loss(
            outputs["main"],
            targets,
        )

        if (
            "deep_2" in outputs
            and "deep_3" in outputs
        ):
            deep_2_loss = self.single_output_loss(
                outputs["deep_2"],
                targets,
            )

            deep_3_loss = self.single_output_loss(
                outputs["deep_3"],
                targets,
            )

            return (
                0.70 * main_loss
                + 0.20 * deep_2_loss
                + 0.10 * deep_3_loss
            )

        if "deep_2" in outputs:
            deep_2_loss = self.single_output_loss(
                outputs["deep_2"],
                targets,
            )

            return (
                0.80 * main_loss
                + 0.20 * deep_2_loss
            )

        if "deep_3" in outputs:
            deep_3_loss = self.single_output_loss(
                outputs["deep_3"],
                targets,
            )

            return (
                0.90 * main_loss
                + 0.10 * deep_3_loss
            )

        return main_loss


def build_criterion(
    experiment: ExperimentConfig,
) -> nn.Module:

    if experiment.loss_name == "dice_focal":

        criterion = DiceFocalSegmentationLoss(
            dice_weight=0.50,
            focal_weight=0.50,
        )

    elif (
        experiment.loss_name
        == "dice_focal_tversky"
    ):

        criterion = (
            DiceFocalTverskySegmentationLoss(
                dice_weight=0.25,
                focal_weight=0.15,
                tversky_weight=0.60,
                tversky_alpha=0.30,
                tversky_beta=0.70,
            )
        )

    else:
        raise ValueError(
            "Unsupported loss_name: "
            f"{experiment.loss_name}"
        )

    return criterion


print("Available loss configurations:")
print("1. dice_focal")
print("2. dice_focal_tversky")

In [ ]:
# Cell 26 - Per-image metric computation

@torch.no_grad()
def batch_metric_tensors(
    logits: torch.Tensor,
    targets: torch.Tensor,
    threshold: float = 0.5,
    epsilon: float = 1e-7,
) -> Dict[str, torch.Tensor]:

    predictions = (
        torch.sigmoid(logits) >= threshold
    ).float()

    predictions = predictions.flatten(start_dim=1)
    targets = targets.flatten(start_dim=1)

    true_positive = (predictions * targets).sum(dim=1)
    false_positive = (
        predictions * (1 - targets)
    ).sum(dim=1)
    false_negative = (
        (1 - predictions) * targets
    ).sum(dim=1)
    true_negative = (
        (1 - predictions) * (1 - targets)
    ).sum(dim=1)

    target_positive = targets.sum(dim=1) > 0
    prediction_positive = predictions.sum(dim=1) > 0

    dice = (
        2 * true_positive + epsilon
    ) / (
        2 * true_positive
        + false_positive
        + false_negative
        + epsilon
    )

    iou = (
        true_positive + epsilon
    ) / (
        true_positive
        + false_positive
        + false_negative
        + epsilon
    )

    precision = (
        true_positive + epsilon
    ) / (
        true_positive + false_positive + epsilon
    )

    recall = (
        true_positive + epsilon
    ) / (
        true_positive + false_negative + epsilon
    )

    specificity = (
        true_negative + epsilon
    ) / (
        true_negative + false_positive + epsilon
    )

    false_positive_empty = (
        prediction_positive & (~target_positive)
    ).float()

    return {
        "dice": dice,
        "iou": iou,
        "precision": precision,
        "recall": recall,
        "specificity": specificity,
        "target_positive": target_positive,
        "false_positive_empty": false_positive_empty,
    }

In [ ]:
# Cell 27 - Metric accumulator

class SegmentationMetricAccumulator:
    def __init__(self):
        self.values: Dict[str, List[torch.Tensor]] = {
            "dice": [],
            "iou": [],
            "precision": [],
            "recall": [],
            "specificity": [],
            "target_positive": [],
            "false_positive_empty": [],
        }

    def update(
        self,
        logits: torch.Tensor,
        targets: torch.Tensor,
    ) -> None:

        batch_values = batch_metric_tensors(
            logits=logits,
            targets=targets,
            threshold=cfg.threshold,
        )

        for key, value in batch_values.items():
            self.values[key].append(
                value.detach().cpu()
            )

    def compute(self) -> Dict[str, float]:
        values = {
            key: torch.cat(tensors)
            for key, tensors in self.values.items()
        }

        positive_mask = values["target_positive"].bool()
        empty_mask = ~positive_mask

        def safe_mean(tensor: torch.Tensor) -> float:
            if tensor.numel() == 0:
                return float("nan")
            return float(tensor.float().mean().item())

        metrics = {
            "dice_all": safe_mean(values["dice"]),
            "dice_tumor": safe_mean(values["dice"][positive_mask]),
            "dice_empty": safe_mean(values["dice"][empty_mask]),

            "iou_all": safe_mean(values["iou"]),
            "iou_tumor": safe_mean(values["iou"][positive_mask]),

            "precision_all": safe_mean(values["precision"]),
            "recall_tumor": safe_mean(values["recall"][positive_mask]),
            "specificity_all": safe_mean(values["specificity"]),

            "empty_false_positive_rate": safe_mean(
                values["false_positive_empty"][empty_mask]
            ),

            "num_images": int(values["dice"].numel()),
            "num_tumor_images": int(positive_mask.sum().item()),
            "num_empty_images": int(empty_mask.sum().item()),
        }

        return metrics

In [ ]:
# Cell 28 - Average meter

class AverageMeter:
    def __init__(self):
        self.reset()

    def reset(self) -> None:
        self.total = 0.0
        self.count = 0

    def update(
        self,
        value: float,
        number: int = 1,
    ) -> None:
        self.total += value * number
        self.count += number

    @property
    def average(self) -> float:
        return self.total / max(self.count, 1)

In [ ]:
# Cell 29 - One training epoch

def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    scaler,
    criterion: nn.Module,
) -> Dict[str, float]:

    model.train()

    loss_meter = AverageMeter()
    metric_accumulator = SegmentationMetricAccumulator()

    progress = tqdm(loader, desc="Train", leave=False)

    for batch in progress:
        images = batch["image"].to(
            cfg.device,
            non_blocking=True,
        )
        masks = batch["mask"].to(
            cfg.device,
            non_blocking=True,
        )

        optimizer.zero_grad(set_to_none=True)

        amp_enabled = (
            cfg.use_amp
            and torch.cuda.is_available()
        )

        with torch.autocast(
            device_type="cuda",
            enabled=amp_enabled,
        ):
            outputs = model(images)
            loss = criterion(outputs, masks)

        scaler.scale(loss).backward()

        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=cfg.gradient_clip,
        )

        scaler.step(optimizer)
        scaler.update()

        batch_size = images.size(0)
        loss_meter.update(loss.item(), batch_size)

        metric_accumulator.update(
            outputs["main"],
            masks,
        )

        progress.set_postfix(
            loss=f"{loss_meter.average:.4f}"
        )

    metrics = metric_accumulator.compute()
    metrics["loss"] = loss_meter.average

    return metrics

In [ ]:
# Cell 30 - Validation or test epoch

@torch.no_grad()
def evaluate_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: Optional[nn.Module] = None,
    description: str = "Evaluate",
) -> Dict[str, float]:

    model.eval()

    loss_meter = AverageMeter()
    metric_accumulator = SegmentationMetricAccumulator()

    for batch in tqdm(loader, desc=description, leave=False):
        images = batch["image"].to(
            cfg.device,
            non_blocking=True,
        )
        masks = batch["mask"].to(
            cfg.device,
            non_blocking=True,
        )

        outputs = model(images)

        if criterion is not None:
            loss = criterion(outputs, masks)
            loss_meter.update(
                loss.item(),
                images.size(0),
            )

        metric_accumulator.update(
            outputs["main"],
            masks,
        )

    metrics = metric_accumulator.compute()

    if criterion is not None:
        metrics["loss"] = loss_meter.average

    return metrics

In [ ]:
# Cell 31 - Checkpoint utilities

def save_checkpoint(
    path: Path,
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    scheduler,
    scaler,
    epoch: int,
    validation_metrics: Dict[str, float],
    experiment: ExperimentConfig,
) -> None:

    checkpoint = {
        "epoch": epoch,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "scaler_state": scaler.state_dict(),
        "validation_metrics": validation_metrics,
        "experiment": asdict(experiment),
        "global_config": asdict(cfg),
    }

    torch.save(checkpoint, path)


def load_model_checkpoint(
    model: nn.Module,
    checkpoint_path: Path,
) -> Dict:

    checkpoint = torch.load(
        checkpoint_path,
        map_location=cfg.device,
    )

    model.load_state_dict(checkpoint["model_state"])

    return checkpoint

In [ ]:
## Cell 32 - Train a complete experiment

def train_experiment(
    experiment: ExperimentConfig,
) -> Dict:

    seed_everything(cfg.seed)

    experiment_directory = (
        Path(cfg.results_dir)
        / experiment.name
    )

    experiment_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    with open(
        experiment_directory
        / "experiment_config.json",
        "w",
    ) as file:

        json.dump(
            {
                "experiment": asdict(
                    experiment
                ),
                "global": asdict(cfg),
            },
            file,
            indent=4,
        )

    (
        train_loader,
        validation_loader,
        test_loader,
    ) = build_dataloaders(
        experiment
    )

    model = build_model(
        experiment
    ).to(cfg.device)

    # Build the loss specified by this experiment.
    criterion = build_criterion(
        experiment
    ).to(cfg.device)

    print(
        f"Experiment: {experiment.name}"
    )

    print(
        f"Loss: {experiment.loss_name}"
    )

    parameter_count = sum(
        parameter.numel()
        for parameter in model.parameters()
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=cfg.learning_rate,
        weight_decay=cfg.weight_decay,
    )

    scheduler = (
        torch.optim.lr_scheduler
        .ReduceLROnPlateau(
            optimizer,
            mode="max",
            factor=cfg.scheduler_factor,
            patience=cfg.scheduler_patience,
        )
    )

    amp_enabled = (
        cfg.use_amp
        and torch.cuda.is_available()
    )

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=amp_enabled,
    )

    checkpoint_path = (
        experiment_directory
        / "best_model.pt"
    )

    best_validation_dice = -np.inf
    best_epoch = 0
    epochs_without_improvement = 0

    history = []

    start_time = time.time()

    for epoch in range(
        1,
        cfg.epochs + 1,
    ):

        print(
            f"\n[{experiment.name}] "
            f"Epoch {epoch}/{cfg.epochs}"
        )

        train_metrics = train_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            scaler=scaler,
            criterion=criterion,
        )

        validation_metrics = evaluate_epoch(
            model=model,
            loader=validation_loader,
            criterion=criterion,
            description="Validation",
        )

        monitor_score = (
            validation_metrics[
                "dice_tumor"
            ]
        )

        # Use Dice over all images when no positive
        # validation image is available.
        if np.isnan(monitor_score):
            monitor_score = (
                validation_metrics[
                    "dice_all"
                ]
            )

        scheduler.step(
            monitor_score
        )

        current_learning_rate = (
            optimizer
            .param_groups[0]["lr"]
        )

        epoch_record = {
            "epoch": epoch,
            "learning_rate": (
                current_learning_rate
            ),
            **{
                f"train_{key}": value
                for key, value
                in train_metrics.items()
            },
            **{
                f"validation_{key}": value
                for key, value
                in validation_metrics.items()
            },
        }

        history.append(
            epoch_record
        )

        pd.DataFrame(
            history
        ).to_csv(
            experiment_directory
            / "history.csv",
            index=False,
        )

        print(
            f"Train loss="
            f"{train_metrics['loss']:.4f} | "
            f"Val loss="
            f"{validation_metrics['loss']:.4f} | "
            f"Val Dice tumor="
            f"{validation_metrics['dice_tumor']:.4f}"
        )

        if (
            monitor_score
            > best_validation_dice
        ):

            best_validation_dice = (
                monitor_score
            )

            best_epoch = epoch
            epochs_without_improvement = 0

            save_checkpoint(
                path=checkpoint_path,
                model=model,
                optimizer=optimizer,
                scheduler=scheduler,
                scaler=scaler,
                epoch=epoch,
                validation_metrics=(
                    validation_metrics
                ),
                experiment=experiment,
            )

        else:
            epochs_without_improvement += 1

        if (
            epochs_without_improvement
            >= cfg.early_stopping_patience
        ):
            print("Early stopping.")
            break

    training_minutes = (
        time.time() - start_time
    ) / 60

    load_model_checkpoint(
        model=model,
        checkpoint_path=checkpoint_path,
    )

    test_metrics = evaluate_epoch(
        model=model,
        loader=test_loader,
        criterion=criterion,
        description="Test",
    )

    result = {
        "experiment": experiment.name,
        "loss_name": experiment.loss_name,

        "residual": (
            experiment.use_residual
        ),
        "cbam": (
            experiment.use_cbam
        ),
        "augmentation": (
            experiment.use_augmentation
        ),
        "aspp": (
            experiment.use_aspp
        ),
        "deep_supervision": (
            experiment.use_deep_supervision
        ),

        "parameters": parameter_count,
        "best_epoch": best_epoch,
        "training_minutes": (
            training_minutes
        ),

        "best_validation_dice_tumor": (
            best_validation_dice
        ),

        **{
            f"test_{key}": value
            for key, value
            in test_metrics.items()
        },
    }

    with open(
        experiment_directory
        / "result.json",
        "w",
    ) as file:

        json.dump(
            result,
            file,
            indent=4,
        )

    del model
    del criterion
    del optimizer
    del scheduler
    del scaler
    del train_loader
    del validation_loader
    del test_loader

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result

In [ ]:
# Cell 33 - Main architecture experiments and loss comparison

MAIN_EXPERIMENTS = [
    ExperimentConfig(
        name="unet",
        use_residual=False,
        use_cbam=False,
        use_augmentation=False,
        use_aspp=True,
        use_deep_supervision=True,
        loss_name="dice_focal",
    ),
    ExperimentConfig(
        name="unet_aug",
        use_residual=False,
        use_cbam=False,
        use_augmentation=True,
        use_aspp=True,
        use_deep_supervision=True,
        loss_name="dice_focal",
    ),
    ExperimentConfig(
        name="residual_unet",
        use_residual=True,
        use_cbam=False,
        use_augmentation=False,
        use_aspp=True,
        use_deep_supervision=True,
        loss_name="dice_focal",
    ),
    ExperimentConfig(
        name="residual_unet_aug",
        use_residual=True,
        use_cbam=False,
        use_augmentation=True,
        use_aspp=True,
        use_deep_supervision=True,
        loss_name="dice_focal",
    ),
    ExperimentConfig(
        name="cbam_unet",
        use_residual=False,
        use_cbam=True,
        use_augmentation=False,
        use_aspp=True,
        use_deep_supervision=True,
        loss_name="dice_focal",
    ),
    ExperimentConfig(
        name="cbam_unet_aug",
        use_residual=False,
        use_cbam=True,
        use_augmentation=True,
        use_aspp=True,
        use_deep_supervision=True,
        loss_name="dice_focal",
    ),
    ExperimentConfig(
        name="residual_cbam_unet",
        use_residual=True,
        use_cbam=True,
        use_augmentation=False,
        use_aspp=True,
        use_deep_supervision=True,
        loss_name="dice_focal",
    ),
    ExperimentConfig(
        name="residual_cbam_unet_aug",
        use_residual=True,
        use_cbam=True,
        use_augmentation=True,
        use_aspp=True,
        use_deep_supervision=True,
        loss_name="dice_focal",
    ),
]


# The architecture selected as best in the previous
# architecture ablation is evaluated again with Tversky.
TVERSKY_LOSS_EXPERIMENT = ExperimentConfig(
    name=(
        "residual_cbam_unet_aug"
        "_dice_focal_tversky"
    ),
    use_residual=True,
    use_cbam=True,
    use_augmentation=True,
    use_aspp=True,
    use_deep_supervision=True,
    loss_name="dice_focal_tversky",
)


LOSS_COMPARISON_EXPERIMENTS = [
    MAIN_EXPERIMENTS[-1],
    TVERSKY_LOSS_EXPERIMENT,
]


print("Main architecture experiments:")

display(
    pd.DataFrame([
        asdict(experiment)
        for experiment
        in MAIN_EXPERIMENTS
    ])
)

print("Controlled loss comparison:")

display(
    pd.DataFrame([
        asdict(experiment)
        for experiment
        in LOSS_COMPARISON_EXPERIMENTS
    ])
)

In [ ]:
# Cell 34 - Run selected experiments

# Recommended setting:
# Run the baseline best architecture and its
# recall-oriented Tversky counterpart.
EXPERIMENTS_TO_RUN = (
    LOSS_COMPARISON_EXPERIMENTS
)

# To run the complete architecture ablation instead,
# use the following line:
# EXPERIMENTS_TO_RUN = MAIN_EXPERIMENTS


main_results_path = (
    Path(cfg.results_dir)
    / "main_ablation_results.csv"
)

if main_results_path.exists():

    main_results_df = pd.read_csv(
        main_results_path
    )

    # Older result files do not contain loss_name.
    if (
        "loss_name"
        not in main_results_df.columns
    ):
        main_results_df[
            "loss_name"
        ] = "dice_focal"

    main_results = (
        main_results_df
        .to_dict("records")
    )

else:
    main_results = []


completed_experiments = {
    result["experiment"]
    for result in main_results
}


for experiment in EXPERIMENTS_TO_RUN:

    if (
        experiment.name
        in completed_experiments
    ):
        print(
            "Skipping completed experiment: "
            f"{experiment.name}"
        )
        continue

    result = train_experiment(
        experiment
    )

    main_results.append(
        result
    )

    completed_experiments.add(
        experiment.name
    )

    pd.DataFrame(
        main_results
    ).to_csv(
        main_results_path,
        index=False,
    )


main_results_df = pd.DataFrame(
    main_results
)

if (
    "loss_name"
    not in main_results_df.columns
):
    main_results_df[
        "loss_name"
    ] = "dice_focal"

main_results_df[
    "loss_name"
] = (
    main_results_df[
        "loss_name"
    ]
    .fillna("dice_focal")
)


display_columns = [
    "experiment",
    "loss_name",
    "residual",
    "cbam",
    "augmentation",
    "aspp",
    "deep_supervision",
    "test_dice_tumor",
    "test_iou_tumor",
    "test_precision_all",
    "test_recall_tumor",
    "test_specificity_all",
    "test_empty_false_positive_rate",
]


display(
    main_results_df[
        [
            column
            for column in display_columns
            if column
            in main_results_df.columns
        ]
    ]
    .sort_values(
        "test_dice_tumor",
        ascending=False,
    )
    .round(4)
)

In [ ]:
# Cell 35 - Choose the best baseline architecture

main_results_df = pd.read_csv(
    Path(cfg.results_dir)
    / "main_ablation_results.csv"
)

if (
    "loss_name"
    not in main_results_df.columns
):
    main_results_df[
        "loss_name"
    ] = "dice_focal"

main_results_df[
    "loss_name"
] = (
    main_results_df[
        "loss_name"
    ]
    .fillna("dice_focal")
)


baseline_results_df = (
    main_results_df[
        main_results_df[
            "loss_name"
        ] == "dice_focal"
    ]
    .copy()
)


best_row = (
    baseline_results_df
    .sort_values(
        "test_dice_tumor",
        ascending=False,
    )
    .iloc[0]
)


best_main_experiment = next(
    experiment
    for experiment
    in MAIN_EXPERIMENTS
    if (
        experiment.name
        == best_row["experiment"]
    )
)


print("Best baseline architecture:")
print(asdict(best_main_experiment))

print(
    "Baseline tumor Dice:",
    float(
        best_row[
            "test_dice_tumor"
        ]
    ),
)

In [ ]:
# Cell 36 - ASPP and Deep Supervision ablation on the best main model

SECONDARY_EXPERIMENTS = [
    replace(
        best_main_experiment,
        name=f"{best_main_experiment.name}_no_aspp_no_ds",
        use_aspp=False,
        use_deep_supervision=False,
    ),
    replace(
        best_main_experiment,
        name=f"{best_main_experiment.name}_aspp_only",
        use_aspp=True,
        use_deep_supervision=False,
    ),
    replace(
        best_main_experiment,
        name=f"{best_main_experiment.name}_ds_only",
        use_aspp=False,
        use_deep_supervision=True,
    ),
    replace(
        best_main_experiment,
        name=f"{best_main_experiment.name}_aspp_ds",
        use_aspp=True,
        use_deep_supervision=True,
    ),
]

pd.DataFrame([
    asdict(experiment)
    for experiment in SECONDARY_EXPERIMENTS
])

In [ ]:
# Cell 37 - Run secondary ablation

secondary_results_path = (
    Path(cfg.results_dir)
    / "aspp_deep_supervision_ablation.csv"
)

if secondary_results_path.exists():
    secondary_results = (
        pd.read_csv(secondary_results_path)
        .to_dict("records")
    )
else:
    secondary_results = []

completed_experiments = {
    result["experiment"]
    for result in secondary_results
}

for experiment in SECONDARY_EXPERIMENTS:
    if experiment.name in completed_experiments:
        print(
            f"Skipping completed experiment: "
            f"{experiment.name}"
        )
        continue

    result = train_experiment(experiment)
    secondary_results.append(result)

    pd.DataFrame(secondary_results).to_csv(
        secondary_results_path,
        index=False,
    )

display(
    pd.DataFrame(secondary_results)
    .sort_values(
        "test_dice_tumor",
        ascending=False,
    )
    .round(4)
)

In [ ]:
# Cell 38 - Final comparison table

result_files = [
    (
        Path(cfg.results_dir)
        / "main_ablation_results.csv"
    ),
    (
        Path(cfg.results_dir)
        / "aspp_deep_supervision_ablation.csv"
    ),
]


available_tables = []

for path in result_files:

    if not path.exists():
        continue

    table = pd.read_csv(
        path
    )

    # Results generated before loss configuration
    # was added correspond to the baseline loss.
    if (
        "loss_name"
        not in table.columns
    ):
        table[
            "loss_name"
        ] = "dice_focal"

    table[
        "loss_name"
    ] = (
        table[
            "loss_name"
        ]
        .fillna("dice_focal")
    )

    available_tables.append(
        table
    )


if len(available_tables) == 0:
    raise FileNotFoundError(
        "No experiment result files were found."
    )


final_results = (
    pd.concat(
        available_tables,
        ignore_index=True,
    )
    .drop_duplicates(
        subset=[
            "experiment",
            "loss_name",
        ],
        keep="last",
    )
)


comparison_columns = [
    "experiment",
    "loss_name",
    "residual",
    "cbam",
    "augmentation",
    "aspp",
    "deep_supervision",
    "parameters",
    "best_epoch",
    "training_minutes",
    "test_dice_all",
    "test_dice_tumor",
    "test_dice_empty",
    "test_iou_tumor",
    "test_precision_all",
    "test_recall_tumor",
    "test_specificity_all",
    "test_empty_false_positive_rate",
]


comparison_columns = [
    column
    for column in comparison_columns
    if column in final_results.columns
]


final_comparison = (
    final_results[
        comparison_columns
    ]
    .sort_values(
        [
            "test_dice_tumor",
            "test_recall_tumor",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)


final_comparison.to_csv(
    Path(cfg.results_dir)
    / "final_comparison.csv",
    index=False,
)


display(
    final_comparison.round(4)
)


print("Loss comparison for the selected architecture:")

selected_architecture_comparison = (
    final_comparison[
        (
            final_comparison[
                "residual"
            ] == True
        )
        & (
            final_comparison[
                "cbam"
            ] == True
        )
        & (
            final_comparison[
                "augmentation"
            ] == True
        )
        & (
            final_comparison[
                "aspp"
            ] == True
        )
        & (
            final_comparison[
                "deep_supervision"
            ] == True
        )
    ]
    [
        [
            "experiment",
            "loss_name",
            "test_dice_tumor",
            "test_iou_tumor",
            "test_precision_all",
            "test_recall_tumor",
            "test_specificity_all",
            "test_empty_false_positive_rate",
        ]
    ]
    .sort_values(
        "test_recall_tumor",
        ascending=False,
    )
)


display(
    selected_architecture_comparison
    .round(4)
)

In [ ]:
# Cell 39 - Plot training history of one experiment

def plot_experiment_history(
    experiment_name: str,
) -> None:

    history_path = (
        Path(cfg.results_dir)
        / experiment_name
        / "history.csv"
    )

    history = pd.read_csv(history_path)

    plt.figure(figsize=(8, 5))
    plt.plot(
        history["epoch"],
        history["train_loss"],
        label="Train loss",
    )
    plt.plot(
        history["epoch"],
        history["validation_loss"],
        label="Validation loss",
    )
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(experiment_name)
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

    plt.figure(figsize=(8, 5))
    plt.plot(
        history["epoch"],
        history["train_dice_tumor"],
        label="Train Dice tumor",
    )
    plt.plot(
        history["epoch"],
        history["validation_dice_tumor"],
        label="Validation Dice tumor",
    )
    plt.xlabel("Epoch")
    plt.ylabel("Dice")
    plt.title(experiment_name)
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()


# Example:
plot_experiment_history(
    final_comparison.iloc[0][
        "experiment"
    ]
)

In [ ]:
# Cell 40 - Load the final selected model for inference

def load_trained_experiment(
    experiment: ExperimentConfig,
) -> nn.Module:

    model = build_model(
        experiment
    ).to(cfg.device)

    checkpoint_path = (
        Path(cfg.results_dir)
        / experiment.name
        / "best_model.pt"
    )

    if not checkpoint_path.exists():
        raise FileNotFoundError(
            f"Checkpoint was not found: "
            f"{checkpoint_path}"
        )

    load_model_checkpoint(
        model=model,
        checkpoint_path=checkpoint_path,
    )

    model.eval()

    return model


# Select the best experiment from Cell 38.
best_final_row = (
    final_comparison
    .sort_values(
        [
            "test_dice_tumor",
            "test_recall_tumor",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .iloc[0]
)


best_final_experiment = ExperimentConfig(
    name=str(
        best_final_row["experiment"]
    ),
    use_residual=bool(
        best_final_row["residual"]
    ),
    use_cbam=bool(
        best_final_row["cbam"]
    ),
    use_augmentation=bool(
        best_final_row["augmentation"]
    ),
    use_aspp=bool(
        best_final_row["aspp"]
    ),
    use_deep_supervision=bool(
        best_final_row[
            "deep_supervision"
        ]
    ),
    loss_name=str(
        best_final_row["loss_name"]
    ),
)


# Build the test loader for the selected experiment.
_, _, test_loader = build_dataloaders(
    best_final_experiment
)


# Load the checkpoint of the selected experiment.
trained_model = load_trained_experiment(
    best_final_experiment
)


trained_model.eval()


checkpoint_path = (
    Path(cfg.results_dir)
    / best_final_experiment.name
    / "best_model.pt"
)


print("=" * 70)
print("Final model loaded successfully")
print("=" * 70)
print(
    "Experiment:",
    best_final_experiment.name,
)
print(
    "Loss:",
    best_final_experiment.loss_name,
)
print(
    "Residual:",
    best_final_experiment.use_residual,
)
print(
    "CBAM:",
    best_final_experiment.use_cbam,
)
print(
    "Augmentation:",
    best_final_experiment.use_augmentation,
)
print(
    "ASPP:",
    best_final_experiment.use_aspp,
)
print(
    "Deep supervision:",
    best_final_experiment.use_deep_supervision,
)
print(
    "Checkpoint:",
    checkpoint_path,
)
print(
    "Checkpoint exists:",
    checkpoint_path.exists(),
)
print("=" * 70)

In [ ]:
# Cell 41 - Visualize tumor and empty predictions

@torch.no_grad()
def show_representative_predictions(
    model: nn.Module,
    loader: DataLoader,
    threshold: float = 0.5,
    number_of_empty_samples: int = 3,
) -> pd.DataFrame:

    model.eval()

    tumor_samples = []
    empty_samples = []

    for batch_index, batch in enumerate(loader):

        images = batch["image"].to(cfg.device)
        masks = batch["mask"].to(cfg.device)

        outputs = model(images)

        if isinstance(outputs, dict):
            logits = outputs["main"]
        else:
            logits = outputs

        probabilities = torch.sigmoid(logits)
        predictions = (probabilities >= threshold).float()

        for sample_index in range(images.size(0)):

            image = images[sample_index].detach().cpu()
            mask = masks[sample_index, 0].detach().cpu()

            probability = (
                probabilities[sample_index, 0]
                .detach()
                .cpu()
            )

            prediction = (
                predictions[sample_index, 0]
                .detach()
                .cpu()
            )

            ground_truth_pixels = int(mask.sum().item())
            predicted_pixels = int(prediction.sum().item())

            common_data = {
                "image": image,
                "mask": mask,
                "probability": probability,
                "prediction": prediction,
                "ground_truth_pixels": ground_truth_pixels,
                "predicted_pixels": predicted_pixels,
                "max_probability": float(
                    probability.max().item()
                ),
                "batch_index": batch_index,
                "sample_index": sample_index,
            }

            # No-tumor sample
            if ground_truth_pixels == 0:

                empty_samples.append({
                    **common_data,
                    "dice": None,
                    "mean_probability_on_tumor": None,
                    "case_type": "No tumor",
                })

                continue

            # Tumor sample metrics
            intersection = float(
                (prediction * mask).sum().item()
            )

            dice = (
                2.0 * intersection + 1e-7
            ) / (
                predicted_pixels
                + ground_truth_pixels
                + 1e-7
            )

            mean_probability_on_tumor = float(
                probability[mask > 0].mean().item()
            )

            tumor_samples.append({
                **common_data,
                "dice": float(dice),
                "mean_probability_on_tumor": (
                    mean_probability_on_tumor
                ),
                "case_type": "Tumor",
            })

    if len(tumor_samples) == 0:
        raise RuntimeError(
            "No tumor-containing samples were found."
        )

    tumor_samples = sorted(
        tumor_samples,
        key=lambda sample: sample["dice"],
    )

    # Prefer correctly classified no-tumor samples
    empty_samples = sorted(
        empty_samples,
        key=lambda sample: (
            sample["predicted_pixels"],
            sample["max_probability"],
        ),
    )

    number_of_tumor_samples = len(tumor_samples)

    worst_sample = tumor_samples[0]

    median_sample = tumor_samples[
        number_of_tumor_samples // 2
    ]

    best_sample = tumor_samples[-1]

    selected_samples = [
        {
            **worst_sample,
            "display_name": "Worst tumor case",
        },
        {
            **median_sample,
            "display_name": "Typical tumor case",
        },
        {
            **best_sample,
            "display_name": "Best tumor case",
        },
    ]

    selected_empty_samples = empty_samples[
        :number_of_empty_samples
    ]

    for index, sample in enumerate(
        selected_empty_samples
    ):
        selected_samples.append({
            **sample,
            "display_name": (
                f"No-tumor case {index + 1}"
            ),
        })

    number_of_rows = len(selected_samples)

    figure, axes = plt.subplots(
        number_of_rows,
        4,
        figsize=(15, 4 * number_of_rows),
        constrained_layout=True,
    )

    if number_of_rows == 1:
        axes = np.expand_dims(
            axes,
            axis=0,
        )

    for row, sample in enumerate(selected_samples):

        image = (
            sample["image"]
            .permute(1, 2, 0)
            .numpy()
        )

        # Remove singleton channel for grayscale images
        if image.shape[-1] == 1:
            image = image[..., 0]

        image_min = image.min()
        image_max = image.max()

        image = (
            image - image_min
        ) / (
            image_max
            - image_min
            + 1e-8
        )

        mask = sample["mask"].numpy()
        probability = sample["probability"].numpy()
        prediction = sample["prediction"].numpy()

        # MRI
        axes[row, 0].imshow(
            image,
            cmap="gray" if image.ndim == 2 else None,
        )

        axes[row, 0].set_title(
            sample["display_name"],
            fontsize=11,
        )

        # Ground truth
        axes[row, 1].imshow(
            mask,
            cmap="gray",
            vmin=0,
            vmax=1,
        )

        axes[row, 1].set_title(
            "Ground truth\n"
            f"Pixels={sample['ground_truth_pixels']}",
            fontsize=11,
        )

        # Probability map
        axes[row, 2].imshow(
            probability,
            cmap="viridis",
            vmin=0,
            vmax=1,
        )

        axes[row, 2].set_title(
            "Probability map\n"
            f"Max={sample['max_probability']:.4f}",
            fontsize=11,
        )

        # Binary prediction
        axes[row, 3].imshow(
            prediction,
            cmap="gray",
            vmin=0,
            vmax=1,
        )

        if sample["dice"] is not None:

            prediction_title = (
                "Prediction\n"
                f"Dice={sample['dice']:.4f} | "
                f"Pixels={sample['predicted_pixels']}"
            )

        else:

            prediction_title = (
                "Prediction\n"
                f"False-positive pixels="
                f"{sample['predicted_pixels']}"
            )

        axes[row, 3].set_title(
            prediction_title,
            fontsize=11,
        )

        for axis in axes[row]:
            axis.axis("off")

    figure.suptitle(
        (
            "Representative Test Predictions "
            f"(Threshold = {threshold:.2f})"
        ),
        fontsize=16,
    )

    plt.show()

    # Numerical summary
    summary_rows = []

    for sample in selected_samples:

        summary_rows.append({
            "case": sample["display_name"],
            "batch_index": sample["batch_index"],
            "sample_index": sample["sample_index"],
            "dice": sample["dice"],
            "ground_truth_pixels": (
                sample["ground_truth_pixels"]
            ),
            "predicted_pixels": (
                sample["predicted_pixels"]
            ),
            "max_probability": (
                sample["max_probability"]
            ),
            "mean_probability_on_tumor": (
                sample["mean_probability_on_tumor"]
            ),
        })

    summary_dataframe = pd.DataFrame(
        summary_rows
    )

    display(summary_dataframe)

    tumor_dice_values = np.array(
        [
            sample["dice"]
            for sample in tumor_samples
        ],
        dtype=float,
    )

    completely_missed_tumors = sum(
        sample["predicted_pixels"] == 0
        for sample in tumor_samples
    )

    missed_tumor_rate = (
        completely_missed_tumors
        / number_of_tumor_samples
    )

    no_tumor_false_positive_cases = sum(
        sample["predicted_pixels"] > 0
        for sample in empty_samples
    )

    print(
        f"Total tumor samples evaluated: "
        f"{number_of_tumor_samples}"
    )

    print(
        f"Completely missed tumors: "
        f"{completely_missed_tumors}"
    )

    print(
        f"Missed-tumor rate: "
        f"{missed_tumor_rate:.2%}"
    )

    print(
        f"Mean tumor Dice: "
        f"{tumor_dice_values.mean():.4f}"
    )

    print(
        f"Median tumor Dice: "
        f"{np.median(tumor_dice_values):.4f}"
    )

    print(
        f"Minimum tumor Dice: "
        f"{tumor_dice_values.min():.4f}"
    )

    print(
        f"Maximum tumor Dice: "
        f"{tumor_dice_values.max():.4f}"
    )

    print(
        f"Total no-tumor samples evaluated: "
        f"{len(empty_samples)}"
    )

    print(
        f"No-tumor samples with false positives: "
        f"{no_tumor_false_positive_cases}"
    )

    return summary_dataframe


# Run Cell 41
representative_summary = (
    show_representative_predictions(
        model=trained_model,
        loader=test_loader,
        threshold=cfg.threshold,
        number_of_empty_samples=3,
    )
)

In [ ]:
# Cell 42 - Multi-seed experiment template for final paper results

FINAL_SEEDS = [42, 123, 2026]

def run_multiple_seeds(
    experiment: ExperimentConfig,
    seeds: List[int] = FINAL_SEEDS,
) -> pd.DataFrame:

    original_seed = cfg.seed
    all_seed_results = []

    for seed in seeds:
        cfg.seed = seed

        seeded_experiment = replace(
            experiment,
            name=f"{experiment.name}_seed_{seed}",
        )

        result = train_experiment(
            seeded_experiment
        )
        result["seed"] = seed

        all_seed_results.append(result)

    cfg.seed = original_seed

    results = pd.DataFrame(all_seed_results)

    results.to_csv(
        Path(cfg.results_dir)
        / f"{experiment.name}_multi_seed.csv",
        index=False,
    )

    return results


# Use only after selecting the final architecture:
#multi_seed_results = run_multiple_seeds(
#    best_final_experiment
#)

In [ ]:
# Cell 43 - Mean ± standard deviation summary

def summarize_multi_seed_results(
    results: pd.DataFrame,
) -> pd.DataFrame:

    metric_columns = [
        "test_dice_tumor",
        "test_iou_tumor",
        "test_recall_tumor",
        "test_specificity_all",
        "test_empty_false_positive_rate",
    ]

    summary_records = []

    for metric in metric_columns:
        mean_value = results[metric].mean()
        standard_deviation = results[metric].std(ddof=1)

        summary_records.append({
            "metric": metric,
            "mean": mean_value,
            "standard_deviation": standard_deviation,
            "formatted": (
                f"{mean_value:.4f} ± "
                f"{standard_deviation:.4f}"
            ),
        })

    return pd.DataFrame(summary_records)


# Example:
#display(summarize_multi_seed_results(multi_seed_results))

In [ ]:
# Cell 44 Find best threshold
@torch.no_grad()
def find_recall_oriented_threshold(
    model: nn.Module,
    loader: DataLoader,
    thresholds=None,
    beta: float = 2.0,
):

    if thresholds is None:
        thresholds = np.arange(
            0.05,
            0.81,
            0.05,
        )

    model.eval()

    all_probabilities = []
    all_masks = []

    for batch in loader:

        images = batch["image"].to(cfg.device)
        masks = batch["mask"].to(cfg.device)

        outputs = model(images)

        if isinstance(outputs, dict):
            logits = outputs["main"]
        else:
            logits = outputs

        probabilities = torch.sigmoid(logits)

        all_probabilities.append(
            probabilities.detach().cpu()
        )

        all_masks.append(
            masks.detach().cpu()
        )

    probabilities = torch.cat(
        all_probabilities,
        dim=0,
    )

    masks = torch.cat(
        all_masks,
        dim=0,
    )

    results = []

    for threshold in thresholds:

        predictions = (
            probabilities >= threshold
        ).float()

        true_positive = (
            predictions * masks
        ).sum().item()

        false_positive = (
            predictions * (1.0 - masks)
        ).sum().item()

        false_negative = (
            (1.0 - predictions) * masks
        ).sum().item()

        precision = (
            true_positive
            / (
                true_positive
                + false_positive
                + 1e-8
            )
        )

        recall = (
            true_positive
            / (
                true_positive
                + false_negative
                + 1e-8
            )
        )

        dice = (
            2.0 * true_positive
            / (
                2.0 * true_positive
                + false_positive
                + false_negative
                + 1e-8
            )
        )

        beta_squared = beta ** 2

        f_beta = (
            (1.0 + beta_squared)
            * precision
            * recall
            / (
                beta_squared * precision
                + recall
                + 1e-8
            )
        )

        results.append({
            "threshold": float(threshold),
            "precision": precision,
            "recall": recall,
            "dice": dice,
            "f_beta": f_beta,
            "false_positive_pixels": (
                int(false_positive)
            ),
            "false_negative_pixels": (
                int(false_negative)
            ),
        })

    return pd.DataFrame(results)

In [ ]:
# Cell 45 - Run validation threshold analysis

_, validation_loader, _ = build_dataloaders(
    best_final_experiment
)

threshold_results = (
    find_recall_oriented_threshold(
        model=trained_model,
        loader=validation_loader,
        beta=2.0,
    )
)

display(
    threshold_results.sort_values(
        "f_beta",
        ascending=False,
    )
)

best_threshold_row = (
    threshold_results
    .sort_values(
        "f_beta",
        ascending=False,
    )
    .iloc[0]
)

selected_threshold = float(
    best_threshold_row["threshold"]
)

print("=" * 60)
print("Recall-oriented validation threshold")
print("=" * 60)
print(
    f"Threshold : "
    f"{selected_threshold:.2f}"
)
print(
    f"Precision : "
    f"{best_threshold_row['precision']:.4f}"
)
print(
    f"Recall    : "
    f"{best_threshold_row['recall']:.4f}"
)
print(
    f"Dice      : "
    f"{best_threshold_row['dice']:.4f}"
)
print(
    f"F2-score  : "
    f"{best_threshold_row['f_beta']:.4f}"
)
print("=" * 60)

In [ ]:
# Cell 46 - Reuse the final model selected in Cell 40

best_model = trained_model
best_test_loader = test_loader
best_test_dataset = best_test_loader.dataset

best_checkpoint_path = (
    Path(cfg.results_dir)
    / best_final_experiment.name
    / "best_model.pt"
)

best_model.eval()

print("=" * 70)
print("Best model prepared for error analysis and Grad-CAM")
print("=" * 70)
print(
    "Experiment:",
    best_final_experiment.name,
)
print(
    "Loss:",
    best_final_experiment.loss_name,
)
print(
    "Checkpoint:",
    best_checkpoint_path,
)
print(
    "Checkpoint exists:",
    best_checkpoint_path.exists(),
)
print("=" * 70)

In [ ]:
# Cell 47 - Sample-level error analysis

@torch.no_grad()
def collect_sample_error_analysis(
    model: nn.Module,
    loader: DataLoader,
    threshold: float,
) -> pd.DataFrame:

    model.eval()

    records = []
    dataset_index = 0

    for batch_index, batch in enumerate(
        tqdm(
            loader,
            desc="Collecting sample-level errors",
        )
    ):

        images = batch["image"].to(cfg.device)
        masks = batch["mask"].to(cfg.device)

        outputs = model(images)

        if isinstance(outputs, dict):
            logits = outputs["main"]
        else:
            logits = outputs

        probabilities = torch.sigmoid(logits)

        predictions = (
            probabilities >= threshold
        ).float()

        for sample_index in range(
            images.size(0)
        ):

            ground_truth = masks[
                sample_index,
                0,
            ]

            prediction = predictions[
                sample_index,
                0,
            ]

            probability = probabilities[
                sample_index,
                0,
            ]

            true_positive = float(
                (
                    prediction * ground_truth
                ).sum().item()
            )

            false_positive = float(
                (
                    prediction
                    * (1.0 - ground_truth)
                ).sum().item()
            )

            false_negative = float(
                (
                    (1.0 - prediction)
                    * ground_truth
                ).sum().item()
            )

            true_negative = float(
                (
                    (1.0 - prediction)
                    * (1.0 - ground_truth)
                ).sum().item()
            )

            ground_truth_pixels = int(
                ground_truth.sum().item()
            )

            predicted_pixels = int(
                prediction.sum().item()
            )

            has_tumor = (
                ground_truth_pixels > 0
            )

            dice = (
                2.0 * true_positive + 1e-7
            ) / (
                2.0 * true_positive
                + false_positive
                + false_negative
                + 1e-7
            )

            iou = (
                true_positive + 1e-7
            ) / (
                true_positive
                + false_positive
                + false_negative
                + 1e-7
            )

            precision = (
                true_positive + 1e-7
            ) / (
                true_positive
                + false_positive
                + 1e-7
            )

            recall = (
                true_positive + 1e-7
            ) / (
                true_positive
                + false_negative
                + 1e-7
            )

            specificity = (
                true_negative + 1e-7
            ) / (
                true_negative
                + false_positive
                + 1e-7
            )

            record = {
                "dataset_index": dataset_index,
                "batch_index": batch_index,
                "sample_index": sample_index,
                "has_tumor": has_tumor,
                "ground_truth_pixels": (
                    ground_truth_pixels
                ),
                "predicted_pixels": (
                    predicted_pixels
                ),
                "true_positive": int(
                    true_positive
                ),
                "false_positive": int(
                    false_positive
                ),
                "false_negative": int(
                    false_negative
                ),
                "true_negative": int(
                    true_negative
                ),
                "dice": float(dice),
                "iou": float(iou),
                "precision": float(precision),
                "recall": float(recall),
                "specificity": float(
                    specificity
                ),
                "maximum_probability": float(
                    probability.max().item()
                ),
                "mean_probability_on_tumor": (
                    float(
                        probability[
                            ground_truth > 0
                        ].mean().item()
                    )
                    if has_tumor
                    else np.nan
                ),
                "completely_missed": (
                    has_tumor
                    and predicted_pixels == 0
                ),
            }

            if "patient" in batch:
                record["patient"] = (
                    batch["patient"][
                        sample_index
                    ]
                )

            if "image_path" in batch:
                record["image_path"] = (
                    batch["image_path"][
                        sample_index
                    ]
                )

            records.append(record)
            dataset_index += 1

    return pd.DataFrame(records)


error_analysis_df = (
    collect_sample_error_analysis(
        model=best_model,
        loader=best_test_loader,
        threshold=cfg.threshold,
    )
)

tumor_error_df = (
    error_analysis_df[
        error_analysis_df["has_tumor"]
    ]
    .copy()
    .reset_index(drop=True)
)

empty_error_df = (
    error_analysis_df[
        ~error_analysis_df["has_tumor"]
    ]
    .copy()
    .reset_index(drop=True)
)

display(
    tumor_error_df
    .sort_values(
        "dice",
        ascending=True,
    )
    .head(10)
)

In [ ]:
# Cell 48 - Error-analysis summary

error_summary = pd.DataFrame({
    "metric": [
        "Tumor-containing samples",
        "Mean tumor Dice",
        "Median tumor Dice",
        "Mean tumor IoU",
        "Mean tumor Precision",
        "Mean tumor Recall",
        "Mean tumor Specificity",
        "Completely missed tumors",
        "Completely missed tumor rate",
        "Mean false-negative pixels",
        "Mean false-positive pixels",
        "No-tumor samples",
        "No-tumor cases with false positives",
    ],
    "value": [
        len(tumor_error_df),
        tumor_error_df["dice"].mean(),
        tumor_error_df["dice"].median(),
        tumor_error_df["iou"].mean(),
        tumor_error_df["precision"].mean(),
        tumor_error_df["recall"].mean(),
        tumor_error_df[
            "specificity"
        ].mean(),
        int(
            tumor_error_df[
                "completely_missed"
            ].sum()
        ),
        tumor_error_df[
            "completely_missed"
        ].mean(),
        tumor_error_df[
            "false_negative"
        ].mean(),
        tumor_error_df[
            "false_positive"
        ].mean(),
        len(empty_error_df),
        int(
            (
                empty_error_df[
                    "false_positive"
                ] > 0
            ).sum()
        ),
    ],
})

display(error_summary)

print("Worst Dice cases:")
display(
    tumor_error_df
    .sort_values("dice")
    .head(10)
)

print("Highest false-negative cases:")
display(
    tumor_error_df
    .sort_values(
        "false_negative",
        ascending=False,
    )
    .head(10)
)

print("Highest false-positive cases:")
display(
    error_analysis_df
    .sort_values(
        "false_positive",
        ascending=False,
    )
    .head(10)
)

In [ ]:
# Cell 49 - Segmentation Grad-CAM implementation

class SegmentationGradCAM:
    """
    Grad-CAM implementation for binary segmentation models.

    The target score is calculated from the main segmentation
    logits inside a user-defined spatial region. The selected
    target layer must produce multi-channel feature maps.
    """

    def __init__(
        self,
        model: nn.Module,
        target_layer: nn.Module,
    ):
        self.model = model
        self.target_layer = target_layer

        self.activations = None
        self.gradients = None

        self.forward_handle = (
            self.target_layer.register_forward_hook(
                self._save_activations
            )
        )

        self.backward_handle = (
            self.target_layer.register_full_backward_hook(
                self._save_gradients
            )
        )

    def _save_activations(
        self,
        module: nn.Module,
        inputs,
        output: torch.Tensor,
    ) -> None:
        """
        Store feature maps produced by the target layer.
        """

        if isinstance(output, (tuple, list)):
            output = output[0]

        if not isinstance(output, torch.Tensor):
            raise TypeError(
                "The Grad-CAM target layer must return "
                "a torch.Tensor."
            )

        self.activations = output

    def _save_gradients(
        self,
        module: nn.Module,
        grad_input,
        grad_output,
    ) -> None:
        """
        Store gradients of the target score with respect
        to the target-layer feature maps.
        """

        if not grad_output:
            self.gradients = None
            return

        gradient = grad_output[0]

        if isinstance(gradient, (tuple, list)):
            gradient = gradient[0]

        self.gradients = gradient

    @staticmethod
    def _extract_main_logits(
        outputs,
    ) -> torch.Tensor:
        """
        Extract the main segmentation logits from different
        model-output formats.
        """

        if isinstance(outputs, dict):

            if "main" not in outputs:
                raise KeyError(
                    "The model output dictionary does not "
                    "contain the 'main' key."
                )

            logits = outputs["main"]

        elif isinstance(outputs, (tuple, list)):

            if len(outputs) == 0:
                raise ValueError(
                    "The model returned an empty sequence."
                )

            logits = outputs[0]

        elif isinstance(outputs, torch.Tensor):
            logits = outputs

        else:
            raise TypeError(
                "Unsupported model-output type: "
                f"{type(outputs)}"
            )

        if logits.ndim != 4:
            raise ValueError(
                "The main segmentation logits must have "
                "shape [B, C, H, W], but received "
                f"{tuple(logits.shape)}."
            )

        return logits

    @staticmethod
    def _prepare_region_mask(
        region_mask: torch.Tensor,
        logits: torch.Tensor,
    ) -> torch.Tensor:
        """
        Convert the region mask to [B, 1, H, W] and resize it
        to match the segmentation-logit resolution.
        """

        if not isinstance(
            region_mask,
            torch.Tensor,
        ):
            region_mask = torch.as_tensor(
                region_mask
            )

        region_mask = region_mask.to(
            device=logits.device,
            dtype=logits.dtype,
        )

        if region_mask.ndim == 2:
            region_mask = region_mask[
                None,
                None,
                :,
                :,
            ]

        elif region_mask.ndim == 3:

            if region_mask.shape[0] == logits.shape[0]:
                region_mask = region_mask[:, None, :, :]

            else:
                region_mask = region_mask[
                    None,
                    :,
                    :,
                    :,
                ]

        elif region_mask.ndim != 4:
            raise ValueError(
                "region_mask must have 2, 3, or 4 "
                "dimensions."
            )

        if region_mask.shape[0] == 1 and logits.shape[0] > 1:
            region_mask = region_mask.expand(
                logits.shape[0],
                -1,
                -1,
                -1,
            )

        if region_mask.shape[0] != logits.shape[0]:
            raise ValueError(
                "The region-mask batch size does not match "
                "the model-output batch size."
            )

        if region_mask.shape[1] != 1:
            region_mask = region_mask.mean(
                dim=1,
                keepdim=True,
            )

        if region_mask.shape[-2:] != logits.shape[-2:]:
            region_mask = F.interpolate(
                region_mask,
                size=logits.shape[-2:],
                mode="nearest",
            )

        region_mask = (
            region_mask > 0
        ).to(
            dtype=logits.dtype
        )

        if region_mask.sum().item() <= 0:
            raise ValueError(
                "The Grad-CAM target region is empty."
            )

        return region_mask

    def generate(
        self,
        image_tensor: torch.Tensor,
        region_mask: torch.Tensor,
    ) -> Dict[str, torch.Tensor]:
        """
        Generate a normalized Grad-CAM map for a selected
        spatial region in the segmentation output.
        """

        self.model.eval()

        if image_tensor.ndim != 4:
            raise ValueError(
                "image_tensor must have shape "
                "[B, C, H, W]."
            )

        image_tensor = image_tensor.to(
            next(self.model.parameters()).device
        )

        image_tensor = image_tensor.contiguous()

        self.model.zero_grad(
            set_to_none=True
        )

        self.activations = None
        self.gradients = None

        with torch.enable_grad():

            outputs = self.model(
                image_tensor
            )

            logits = self._extract_main_logits(
                outputs
            )

            prepared_region_mask = (
                self._prepare_region_mask(
                    region_mask=region_mask,
                    logits=logits,
                )
            )

            # Use the average tumor logit inside the selected
            # region as the Grad-CAM target score.
            target_score = (
                logits
                * prepared_region_mask
            ).sum() / (
                prepared_region_mask.sum()
                + 1e-8
            )

            target_score.backward(
                retain_graph=False
            )

        if self.activations is None:
            raise RuntimeError(
                "The forward hook did not capture "
                "target-layer activations."
            )

        if self.gradients is None:
            raise RuntimeError(
                "The backward hook did not capture "
                "target-layer gradients."
            )

        activations = self.activations
        gradients = self.gradients

        if activations.ndim != 4:
            raise ValueError(
                "The target-layer activations must have "
                "shape [B, C, H, W], but received "
                f"{tuple(activations.shape)}."
            )

        if gradients.ndim != 4:
            raise ValueError(
                "The target-layer gradients must have "
                "shape [B, C, H, W], but received "
                f"{tuple(gradients.shape)}."
            )

        if activations.shape != gradients.shape:
            raise ValueError(
                "Activation and gradient shapes do not "
                "match: "
                f"{tuple(activations.shape)} versus "
                f"{tuple(gradients.shape)}."
            )

        if activations.shape[1] <= 1:
            raise ValueError(
                "The selected Grad-CAM layer has only one "
                "output channel. Select a multi-channel "
                "decoder feature layer instead of a "
                "spatial-attention or output-head layer."
            )

        # Spatially average gradients to obtain one
        # importance weight for each feature channel.
        channel_weights = gradients.mean(
            dim=(2, 3),
            keepdim=True,
        )

        # Combine feature maps using the gradient-derived
        # channel-importance weights.
        raw_cam = (
            channel_weights
            * activations
        ).sum(
            dim=1,
            keepdim=True,
        )

        # Keep positive evidence for the selected tumor target.
        raw_cam = F.relu(
            raw_cam,
            inplace=False,
        )

        raw_cam = F.interpolate(
            raw_cam,
            size=image_tensor.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )

        raw_cam_max = float(
            raw_cam.max()
            .detach()
            .cpu()
            .item()
        )

        raw_cam_mean = float(
            raw_cam.mean()
            .detach()
            .cpu()
            .item()
        )

        raw_cam_std = float(
            raw_cam.std()
            .detach()
            .cpu()
            .item()
        )

        # Normalize every sample independently to [0, 1].
        cam_min = raw_cam.amin(
            dim=(2, 3),
            keepdim=True,
        )

        cam_max = raw_cam.amax(
            dim=(2, 3),
            keepdim=True,
        )

        cam_range = (
            cam_max
            - cam_min
        )

        nearly_zero_samples = (
            cam_range <= 1e-12
        )

        cam = (
            raw_cam
            - cam_min
        ) / (
            cam_range
            + 1e-8
        )

        # Keep degenerate maps exactly zero rather than
        # amplifying numerical noise.
        cam = torch.where(
            nearly_zero_samples,
            torch.zeros_like(cam),
            cam,
        )

        if raw_cam_max <= 1e-12:
            print(
                "Warning: the generated Grad-CAM is nearly "
                "zero. Verify that Cell 50 selects a "
                "multi-channel decoder feature layer."
            )

        elif raw_cam_std <= 1e-12:
            print(
                "Warning: the generated Grad-CAM has almost "
                "no spatial variation. Consider selecting "
                "an earlier decoder layer."
            )

        return {
            "cam": cam.detach(),
            "raw_cam": raw_cam.detach(),
            "logits": logits.detach(),
            "region_mask": (
                prepared_region_mask.detach()
            ),
            "target_score": (
                target_score.detach()
            ),
            "raw_cam_max": raw_cam_max,
            "raw_cam_mean": raw_cam_mean,
            "raw_cam_std": raw_cam_std,
            "activation_shape": tuple(
                activations.shape
            ),
            "gradient_shape": tuple(
                gradients.shape
            ),
        }

    def remove_hooks(
        self,
    ) -> None:
        """
        Remove registered hooks after all Grad-CAM
        visualizations have been generated.
        """

        if self.forward_handle is not None:
            self.forward_handle.remove()
            self.forward_handle = None

        if self.backward_handle is not None:
            self.backward_handle.remove()
            self.backward_handle = None

        self.activations = None
        self.gradients = None

        print(
            "Grad-CAM hooks were removed."
        )

In [ ]:
# Cell 50 - Select a multi-channel decoder feature layer for Grad-CAM

# Grad-CAM must be applied to a multi-channel feature layer,
# not to the single-channel CBAM spatial-attention convolution.

available_modules = dict(
    best_model.named_modules()
)

preferred_gradcam_layers = [
    "decoder2.features.main.3",
    "decoder3.features.main.3",
    "decoder1.features.main.3",
]

gradcam_layer_name = None
gradcam_target_layer = None

for candidate_name in preferred_gradcam_layers:

    if candidate_name in available_modules:

        candidate_module = available_modules[
            candidate_name
        ]

        if (
            isinstance(
                candidate_module,
                nn.Conv2d,
            )
            and candidate_module.out_channels > 1
        ):
            gradcam_layer_name = candidate_name
            gradcam_target_layer = candidate_module
            break


if gradcam_target_layer is None:
    raise RuntimeError(
        "A suitable multi-channel decoder layer "
        "was not found for Grad-CAM."
    )


segmentation_gradcam = SegmentationGradCAM(
    model=best_model,
    target_layer=gradcam_target_layer,
)


print("=" * 70)
print("Grad-CAM target layer selected")
print("=" * 70)
print(
    "Layer name:",
    gradcam_layer_name,
)
print(
    "Layer type:",
    gradcam_target_layer,
)
print(
    "Output channels:",
    gradcam_target_layer.out_channels,
)
print("=" * 70)

In [ ]:
# ============================================================
# Grad-CAM Cell 51 - Generate Grad-CAM for one test sample
# ============================================================

def normalize_image_for_display(
    image_tensor: torch.Tensor,
) -> np.ndarray:

    image = (
        image_tensor
        .detach()
        .cpu()
        .permute(1, 2, 0)
        .numpy()
    )

    image = (
        image - image.min()
    ) / (
        image.max()
        - image.min()
        + 1e-8
    )

    return image.astype(np.float32)


def apply_heatmap_overlay(
    image: np.ndarray,
    heatmap: np.ndarray,
    alpha: float = 0.45,
) -> np.ndarray:

    heatmap_rgb = plt.cm.jet(
        heatmap
    )[..., :3]

    overlay = (
        (1.0 - alpha) * image
        + alpha * heatmap_rgb
    )

    return np.clip(
        overlay,
        0.0,
        1.0,
    )


def generate_gradcam_for_sample(
    dataset_index: int,
    target_mode: str = "ground_truth",
    threshold: float = 0.5,
) -> Dict:

    sample = best_test_dataset[
        int(dataset_index)
    ]

    image_tensor = (
        sample["image"]
        .unsqueeze(0)
        .to(cfg.device)
    )

    ground_truth_tensor = (
        sample["mask"][0]
        .detach()
        .cpu()
    )

    with torch.no_grad():

        outputs = best_model(
            image_tensor
        )

        probability_tensor = torch.sigmoid(
            outputs["main"]
        )[0, 0]

        prediction_tensor = (
            probability_tensor >= threshold
        ).float()

    if target_mode == "ground_truth":

        region_mask = ground_truth_tensor

        if region_mask.sum().item() == 0:
            raise ValueError(
                "This sample has no ground-truth tumor."
            )

    elif target_mode == "prediction":

        region_mask = (
            prediction_tensor
            .detach()
            .cpu()
        )

        if region_mask.sum().item() == 0:

            probability_cpu = (
                probability_tensor
                .detach()
                .cpu()
            )

            percentile_threshold = torch.quantile(
                probability_cpu.flatten(),
                0.95,
            )

            region_mask = (
                probability_cpu
                >= percentile_threshold
            ).float()

    else:
        raise ValueError(
            "target_mode must be either "
            "'ground_truth' or 'prediction'."
        )

    gradcam_result = (
        segmentation_gradcam.generate(
            image_tensor=image_tensor,
            region_mask=region_mask,
        )
    )

    cam = (
        gradcam_result["cam"][0, 0]
        .detach()
        .cpu()
        .numpy()
    )

    image = normalize_image_for_display(
        sample["image"]
    )

    ground_truth = (
        ground_truth_tensor.numpy()
    )

    probability = (
        probability_tensor
        .detach()
        .cpu()
        .numpy()
    )

    prediction = (
        prediction_tensor
        .detach()
        .cpu()
        .numpy()
    )

    overlay = apply_heatmap_overlay(
        image=image,
        heatmap=cam,
        alpha=0.45,
    )

    intersection = float(
        (
            prediction
            * ground_truth
        ).sum()
    )

    dice = (
        2.0 * intersection + 1e-7
    ) / (
        prediction.sum()
        + ground_truth.sum()
        + 1e-7
    )

    false_negative_map = (
        (ground_truth == 1)
        & (prediction == 0)
    ).astype(np.float32)

    false_positive_map = (
        (ground_truth == 0)
        & (prediction == 1)
    ).astype(np.float32)

    return {
        "dataset_index": int(dataset_index),
        "patient": sample["patient"],
        "image_path": sample["image_path"],
        "image": image,
        "ground_truth": ground_truth,
        "probability": probability,
        "prediction": prediction,
        "gradcam": cam,
        "overlay": overlay,
        "false_negative": false_negative_map,
        "false_positive": false_positive_map,
        "dice": float(dice),
        "target_mode": target_mode,
    }

In [ ]:
# ============================================================
# Grad-CAM Cell 52 - Visualize Grad-CAM and segmentation errors
# ============================================================

def show_gradcam_result(
    dataset_index: int,
    target_mode: str = "ground_truth",
    threshold: float = 0.5,
) -> None:

    result = generate_gradcam_for_sample(
        dataset_index=dataset_index,
        target_mode=target_mode,
        threshold=threshold,
    )

    figure, axes = plt.subplots(
        2,
        4,
        figsize=(16, 8),
        constrained_layout=True,
    )

    axes[0, 0].imshow(
        result["image"]
    )
    axes[0, 0].set_title("MRI image")

    axes[0, 1].imshow(
        result["ground_truth"],
        cmap="gray",
        vmin=0,
        vmax=1,
    )
    axes[0, 1].set_title("Ground truth")

    axes[0, 2].imshow(
        result["probability"],
        cmap="viridis",
        vmin=0,
        vmax=1,
    )
    axes[0, 2].set_title("Tumor probability")

    axes[0, 3].imshow(
        result["prediction"],
        cmap="gray",
        vmin=0,
        vmax=1,
    )
    axes[0, 3].set_title(
        f"Prediction\nDice={result['dice']:.4f}"
    )

    axes[1, 0].imshow(
        result["gradcam"],
        cmap="jet",
        vmin=0,
        vmax=1,
    )
    axes[1, 0].set_title("Grad-CAM")

    axes[1, 1].imshow(
        result["overlay"]
    )
    axes[1, 1].set_title("Grad-CAM overlay")

    axes[1, 2].imshow(
        result["false_negative"],
        cmap="Reds",
        vmin=0,
        vmax=1,
    )
    axes[1, 2].set_title(
        "False-negative region\n"
        f"Pixels="
        f"{int(result['false_negative'].sum())}"
    )

    axes[1, 3].imshow(
        result["false_positive"],
        cmap="Blues",
        vmin=0,
        vmax=1,
    )
    axes[1, 3].set_title(
        "False-positive region\n"
        f"Pixels="
        f"{int(result['false_positive'].sum())}"
    )

    for axis in axes.ravel():
        axis.axis("off")

    figure.suptitle(
        (
            f"Experiment: "
            f"{best_final_experiment.name}\n"
            f"Patient: {result['patient']} | "
            f"Target: {target_mode} | "
            f"Threshold: {threshold:.2f}"
        ),
        fontsize=14,
    )

    plt.show()

In [ ]:
# ============================================================
# Grad-CAM Cell 53 - Representative Grad-CAM cases
# ============================================================

sorted_tumor_cases = (
    tumor_error_df
    .sort_values(
        "dice",
        ascending=True,
    )
    .reset_index(drop=True)
)

worst_case_index = int(
    sorted_tumor_cases.iloc[0][
        "dataset_index"
    ]
)

typical_case_index = int(
    sorted_tumor_cases.iloc[
        len(sorted_tumor_cases) // 2
    ]["dataset_index"]
)

best_case_index = int(
    sorted_tumor_cases.iloc[-1][
        "dataset_index"
    ]
)

print("Worst tumor case")
show_gradcam_result(
    dataset_index=worst_case_index,
    target_mode="ground_truth",
    threshold=cfg.threshold,
)

print("Typical tumor case")
show_gradcam_result(
    dataset_index=typical_case_index,
    target_mode="ground_truth",
    threshold=cfg.threshold,
)

print("Best tumor case")
show_gradcam_result(
    dataset_index=best_case_index,
    target_mode="ground_truth",
    threshold=cfg.threshold,
)

In [ ]:
# Cell 54 - Release Grad-CAM hooks

segmentation_gradcam.remove_hooks()

print("Grad-CAM hooks were removed.")